# Navitas — Model Eğitimi (Colab / GPU)

**Nasıl kullanılır:** Yukarıda **Çalışma Zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seç.
Sonra her hücreyi sırayla **Shift+Enter** ile çalıştır. Bir hücre bitmeden diğerine geçme.

Eğitilen model: `yolov8n-seg`, 2 sınıf → `traversable` (açık yol) / `blocked` (kapalı yol)
Veri: **RescueNet + FloodNet** · 100 epoch · ~1-2 saat (GPU)


## Adım 1 — GPU var mı kontrol et
Çıktıda bir GPU (ör. Tesla T4) görmelisin. Görmüyorsan yukarıdan T4 GPU seç.

In [ ]:
!nvidia-smi

## Adım 2 — Projeyi GitHub'dan indir
Çalıştırınca **GitHub token** isteyecek. Token ekranda görünmez (güvenli).
> Token nereden: GitHub → Settings → Developer settings → Personal access tokens.

In [ ]:
from getpass import getpass
GITHUB_TOKEN = getpass("GitHub token yapıştır (görünmez): ")
REPO = "navitas26/navitas"
!git clone https://{GITHUB_TOKEN}@github.com/{REPO}.git
!ls navitas

## Adım 3 — Gerekli kütüphaneleri kur

In [ ]:
!pip install ultralytics datasets pillow tqdm opencv-python-headless -q
print('Kurulum tamam')

## Adım 4 — Google Drive'ı bağla
Veri ve eğitilmiş model Drive'a kaydedilir; Colab kapansa bile kaybolmaz.
Çıkan linke tıklayıp izin ver.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/navitas/datasets', exist_ok=True)
os.makedirs('/content/drive/MyDrive/navitas/runs', exist_ok=True)
os.makedirs('/content/drive/MyDrive/navitas/models', exist_ok=True)
print('Drive bağlandı')

## Adım 5 — Veri setlerini indir: RescueNet + FloodNet
İlk seferinde ~10-20 dk sürer (Drive'a cache'lenir). İki veri seti de iner.

In [ ]:
import os
os.environ["HF_DATASETS_CACHE"] = "/content/drive/MyDrive/navitas/datasets"
%cd /content/navitas/dataset
print("=== RescueNet indiriliyor ===")
!python3 download_and_convert.py
print("=== FloodNet indiriliyor ===")
!python3 download_floodnet.py
print("Veri setleri hazır")

## Adım 6 — Eğitim ayar dosyasını yaz (HER İKİ veri seti dahil)
**Önemli:** Burada hem RescueNet hem FloodNet eğitime katılıyor.

In [ ]:
yaml_content = """train:
  - /content/navitas/dataset/rescuenet_yolo/images/train
  - /content/navitas/dataset/floodnet_yolo/images/train

val:
  - /content/navitas/dataset/rescuenet_yolo/images/val
  - /content/navitas/dataset/floodnet_yolo/images/val

test: /content/navitas/dataset/rescuenet_yolo/images/test

nc: 2
names:
  0: traversable
  1: blocked
"""
with open("/content/navitas/dataset/combined_data.yaml", "w") as f:
    f.write(yaml_content)
print(yaml_content)
print("Ayar dosyasi yazildi (RescueNet + FloodNet)")

## Adım 7 — GPU için eğitim ayarı
Eğitim scripti laptop CPU'su için yazılmış. GPU'ya çeviriyoruz (`device=0`, `batch=16`).

In [ ]:
!sed -i 's/device="cpu"/device=0/' /content/navitas/codes/train_seg.py
!sed -i 's/batch=8,/batch=16,/' /content/navitas/codes/train_seg.py
!grep -nE "device=|batch=" /content/navitas/codes/train_seg.py

## Adım 8 — Eğit (100 epoch · ~1-2 saat)
Bu hücre uzun sürer. İlerlemeyi epoch epoch göreceksin. Sekmeyi kapatma.

In [ ]:
%cd /content/navitas/codes
!python3 train_seg.py --data /content/navitas/dataset/combined_data.yaml --epochs 100

## Adım 9 — Eğitilmiş modeli Drive'a kaydet
Sonuçları (best.pt + grafikler + NCNN) Drive'a kopyalar; sonra rahatça indirirsin.

In [ ]:
import glob, shutil, os
hits = glob.glob('/content/navitas/codes/runs/segment/**/weights/best.pt', recursive=True)
print('Bulunan best.pt:', hits)
for h in hits:
    run_dir = os.path.dirname(os.path.dirname(h))
    name = os.path.basename(run_dir)
    dst = f'/content/drive/MyDrive/navitas/runs/{name}'
    shutil.copytree(run_dir, dst, dirs_exist_ok=True)
    print('Run kopyalandi ->', dst)
for m in glob.glob('/content/navitas/models/*_ncnn_model'):
    dst = f'/content/drive/MyDrive/navitas/models/{os.path.basename(m)}'
    shutil.copytree(m, dst, dirs_exist_ok=True)
    print('NCNN kopyalandi ->', dst)
print('Bitti. Drive > navitas > runs ve models klasorlerine bak.')

## Adım 10 (opsiyonel) — NCNN export'u tekrar üret
Adım 8 sonunda otomatik NCNN export olur. Olmadıysa veya tekrar istersen bunu çalıştır (RPi için).

In [ ]:
from ultralytics import YOLO
import glob
hits = glob.glob('/content/drive/MyDrive/navitas/runs/**/weights/best.pt', recursive=True)
print('Kullanilan model:', hits[0])
model = YOLO(hits[0])
out = model.export(format='ncnn', imgsz=512)
print('NCNN export:', out)